In [60]:
import pandas as pd
import numpy as np
from matplotlib.pyplot import plot

In [61]:
house_prices = pd.read_csv("data/Housing.csv")

### Since there are non-numeric from the dataset we must first convert em into numbers

In [62]:
house_prices.head(10)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000,5750,3,2,4,yes,yes,no,no,yes,1,yes,unfurnished


In [63]:
non_numeric_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

house_prices[non_numeric_cols] = house_prices[non_numeric_cols].apply(lambda col: col.map({'yes': 1, 'no': 0}))

house_prices = pd.get_dummies(house_prices, columns=['furnishingstatus'], drop_first=True) # Drop the first furnishingstatus, it will become the baseline later
bool_cols = house_prices.select_dtypes(include='bool').columns
house_prices[bool_cols] = house_prices[bool_cols].astype(int)
house_prices

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,0,0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,0,0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1,0
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,0,0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,0,1
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,1,0
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,0,1
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,0,0


### Now separate the training data from the label data, the price is the label

In [64]:
print(house_prices.isnull().sum()) # Check if there are any null values first, since wala then proceed

price                              0
area                               0
bedrooms                           0
bathrooms                          0
stories                            0
mainroad                           0
guestroom                          0
basement                           0
hotwaterheating                    0
airconditioning                    0
parking                            0
prefarea                           0
furnishingstatus_semi-furnished    0
furnishingstatus_unfurnished       0
dtype: int64


In [65]:
# X train
X = house_prices.iloc[:, 1:]
# y true
y = house_prices.iloc[:, 0]
# Split the data for training and testing ( same sa digit recognition system ko )
# 80% of the data will be used for the training and the rest is for testing
train_percent_index = int(0.8 * len(X))
X_train, X_test = X[:train_percent_index], X[train_percent_index:]
y_train, y_test = y[:train_percent_index], y[train_percent_index:]
# Shuffle timee
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train.iloc[shuffle_idx]
y_train = y_train.iloc[shuffle_idx]
X_train


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
38,6000,3,1,4,1,1,0,0,1,2,0,0,1
350,3420,2,1,2,1,0,0,1,0,1,0,1,0
97,6400,3,1,1,1,1,1,0,1,1,1,1,0
432,6060,3,1,1,1,1,1,0,0,0,0,0,0
415,4785,3,1,2,1,1,1,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,6670,3,1,3,1,0,1,0,0,0,1,0,1
20,4320,3,1,2,1,0,1,1,0,2,0,1,0
346,2176,2,1,2,1,1,0,0,0,0,1,1,0
330,4050,2,1,2,1,1,1,0,0,0,1,0,1


In [66]:
learning_rate = None
W = []
B = []

layer_sizes = [13, 16, 1] # input, hidden, output
np.shape(X_train)
# Init Weights and Biases
for n in range(len(layer_sizes) - 1):
    _in = layer_sizes[n]
    _out = layer_sizes[n+1]
    W.append(np.random.randn(_in, _out) * np.sqrt(2.0 / _in))
    B.append(np.zeros((1, _out)))

Z = [None] * (len(layer_sizes) -1) # Summation aft mat multiplication is applied
H = [None] * (len(layer_sizes) -1) # Activation
len(W)
# self.__W1 = np.random.randn(784, 256) * np.sqrt(2.0 / 784)
# self.__B1 = np.zeros((1, 256))

2

In [67]:
def relu(x_array):
    return np.maximum(0, x_array)


def relu_derivative(z):
    return (z > 0).astype(float)

def MSE(y_pred, y_true):
    return np.square(np.subtract(y_true, y_pred)).mean()

def MSE_derivative(y_true, y_pred): # Na-cow
    """
    Calculates the gradient of MSE with respect to y_pred.
    Formula: 2/n * (y_pred - y_true)
    """
    n = y_true.size
    return 2 * (y_pred - y_true) / n

def forward_pass(X):
    h = X
    for i in range(len(W)):
        Z[i] = h @ W[i] + B[i]
        H[i] = relu(Z[i])
        h = H[i]
    return h # y_pred

def back_propagation(X_batch, y_pred, y_true):
    """ There is probably a simpler/better way of doing this but since I only had 5 hours of sleep I won't overthink it. """
    dL_dy = MSE_derivative(y_true, y_pred)
    # Could just set Z[1] not Z[-1] but it's just the same, and it's best to get accustomed to it cause in the future if am working with larger input params ts's the way am gonna do it
    dL_dz = dL_dy * relu_derivative(Z[-1]) # Pass the y_pred; could've passed the y_pred var instead of the Z but it's just the same
    dL_dW = [None] * len(W)
    dL_dB = [None] * len(W)
    # Start the loop from the max since ts is not forward pass
    for i in reversed(range(len(W))):
        input_h = X_batch if i == 0 else H[0] # Input activation
        dL_dW[i] = input_h.T @ dL_dz
        dL_dB[i] = np.sum(dL_dz, axis=0, keepdims=True)
        if i > 0:
            dL_dz = (dL_dz @ dL_dW[i].T) * relu_derivative(Z[i-1])

    return dL_dW, dL_dB